# RNN Text Classification (LSTM vs GRU)

## الترتيب / Flow
1. استيراد المكتبات - Import libraries
2. قراءة البيانات - Load dataset
3. Tokenization + padding - Text to sequences
4. تقسيم البيانات - Train/Test split
5. بناء LSTM - Build LSTM model
6. تدريب LSTM - Train and evaluate
7. بناء GRU - Build GRU model
8. تدريب GRU - Train and evaluate
9. مقارنة النماذج - Compare accuracy

In [ ]:
# Step 1) استيراد المكتبات / Import libraries
# pip install tensorflow -q  # uncomment in Colab if needed
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, GRU, Dense, Dropout

In [ ]:
# Step 2) قراءة البيانات / Load dataset
dataset = pd.read_csv('sentiment_reviews.csv')
print(dataset['sentiment'].value_counts())
dataset.head()

In [ ]:
# Step 3) Tokenization + padding / Text to sequences
MAX_WORDS = 2000
MAX_LEN = 20

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(dataset['review'])
sequences = tokenizer.texts_to_sequences(dataset['review'])
X = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')
y = dataset['sentiment'].values
print('X shape:', X.shape)

In [ ]:
# Step 4) تقسيم البيانات / Train-Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)

In [ ]:
# Step 5) بناء LSTM / Build LSTM model
def build_rnn_model(rnn_layer):
    model = Sequential([
        Embedding(input_dim=MAX_WORDS, output_dim=64, input_length=MAX_LEN),
        rnn_layer,
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

lstm_model = build_rnn_model(LSTM(64))
lstm_model.summary()

In [ ]:
# Step 6) تدريب LSTM / Train LSTM
lstm_history = lstm_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=15,
    batch_size=32,
    verbose=1
)
lstm_loss, lstm_acc = lstm_model.evaluate(X_test, y_test, verbose=0)
print(f'LSTM test accuracy: {lstm_acc:.2%}')

In [ ]:
# Step 7) بناء GRU / Build GRU model
gru_model = build_rnn_model(GRU(64))
gru_model.summary()

In [ ]:
# Step 8) تدريب GRU / Train GRU
gru_history = gru_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=15,
    batch_size=32,
    verbose=1
)
gru_loss, gru_acc = gru_model.evaluate(X_test, y_test, verbose=0)
print(f'GRU test accuracy: {gru_acc:.2%}')

In [ ]:
# Step 9) مقارنة + تنبؤ على جملة جديدة / Compare models + sample prediction
print('--- RNN Comparison ---')
print(f'LSTM: {lstm_acc:.2%}')
print(f'GRU:  {gru_acc:.2%}')

sample_reviews = [
    'I loved this product it works perfectly',
    'Terrible quality broke after one day'
]
sample_seq = pad_sequences(tokenizer.texts_to_sequences(sample_reviews), maxlen=MAX_LEN)
lstm_preds = (lstm_model.predict(sample_seq, verbose=0) > 0.5).astype(int).flatten()
gru_preds = (gru_model.predict(sample_seq, verbose=0) > 0.5).astype(int).flatten()

for i, text in enumerate(sample_reviews):
    print(f"\nReview: {text}")
    print(f"LSTM sentiment: {lstm_preds[i]} | GRU sentiment: {gru_preds[i]}")